# Pick documents

An annotation project rarely covers a whole database: you first select the documents worth annotating, such as
the notes of one department over a year. In this tutorial, you turn a query on the sample database into an
annotation file holding an editable copy of the selected notes, without writing the query yourself.

!!! info "Before you start"

    Follow [Browse a database](browse-a-database.md) first: this tutorial starts from its `clinic` source.

## Filter the notes

In the Metanno sidebar, open the `notes` table of the `clinic` source. Type `oncologie` in the box under
`department`, then click <span class="ui-button">Apply source filters</span>: the tab now shows the oncology notes
only.

## Copy the code

Click <span class="ui-button">Copy code</span> in the tab toolbar, then <span class="ui-button">Prepare annotation</span>,
and paste the copied code in a notebook cell. It connects to the database, queries the oncology notes, saves them
to an annotation file and opens it. Its comments describe each step, and the paths are the ones of your own
machine.

Before running it, adapt two lines to your project. Add the radiology notes to the cohort, so that breast imaging
reports are reviewed with the oncology notes, and give the file a meaningful name:

```diff
-    .filter(_['department'] == 'oncologie')
+    .filter(_['department'].isin(['oncologie', 'radiologie']))
...
-destination = Path('/home/you/views') / 'stores' / 'annotation.pret'
+destination = Path('/home/you/views') / 'stores' / 'oncology.pret'
```

With these changes, the code reads as below, with your own paths instead of `/home/you`. Paste your version in
the empty cell under it:

```python { .no-exec }
from pathlib import Path

import ibis
from ibis import _

from pret import DataFrame, create_store, load_store, save_store
from metanno import view_opener


config_path = '/home/you'

# Connections of metanno.yaml, rerun this cell if a connection is lost
clinic = ibis.connect('duckdb:///home/you/clinic.duckdb')

# Queries select source records, change the filters and limits for your cohort
notes = (
    clinic.table('notes')
    .filter(_['department'].isin(['oncologie', 'radiologie']))
    .limit(500)
)

# Tables maps the prepared data to the Store keys expected by the view recipe
tables = {
    'notes': DataFrame(notes),
}

# Parameters are keyword arguments for the view recipe, they do not change the source queries
parameters = {}

# Save editable copies of the query results and added fields, without changing the source database
destination = Path('/home/you/views') / 'stores' / 'oncology.pret'
# Annotation views copy every table into the page so filters and edits apply at once, up to 50 MB.
# The options are saved with the Store and used again when it is reopened
options = {'dataframe_sync': 'full', 'dataframe_max_bytes': 50 * 1024 * 1024}
save_store(create_store(tables, sync=True), destination, metadata={
    'view': 'table', 'parameters': parameters, 'config_path': config_path, **options,
})
store = load_store(destination, **options)
view_opener(store)
```

Run it, then click <span class="ui-button">Open with Metanno</span>: the four selected notes open in an editable
tab. `oncology.pret` also appears under **Annotations** in the sidebar, where you or a colleague can reopen it
later.

!!! info "A copy, not the database"

    The annotation file holds a copy of the selected notes: edits go to `oncology.pret`, and your database is never
    modified. The copy is limited to the 500 first rows of the query, raise `.limit(500)` when your cohort is
    larger, and keep in mind that the whole selection is loaded in the annotation tab.

??? tip "Only exploring?"

    <span class="ui-button">Copy code</span> → <span class="ui-button">Customize query</span> writes the same query
    without saving anything, to continue exploring the data in Python, for instance with pandas or Ibis.

## Next

The notes are ready, but nothing is annotated yet. [Pre-annotate](pre-annotate.md) them with EDS-NLP, so that
annotators check entities instead of finding them all by hand.